# Rainbow DQN on MinAtar

Phase 0 baseline: trains the Rainbow agent (double Q-learning, dueling architecture, categorical/C51 value distribution, prioritized replay, n-step returns, noisy-net exploration) from `envs/`, `models/`, and `agents/` on MinAtar, with a training loop and results plot below.

Runs on Colab (with Drive-backed checkpointing) or locally. This is the no-auxiliary-loss baseline everything in Phase 1+ gets compared against -- see `train_spr_minatar.ipynb`.

In [1]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_DIR = "/content/RLresearch"
    if not os.path.exists(REPO_DIR):
        # push local changes before opening this in Colab, or the clone below will be stale
        !git clone https://github.com/Bu3of/RLresearch.git {REPO_DIR}
    %cd {REPO_DIR}
    !pip install -q -r requirements.txt

    CHECKPOINT_DIR = "/content/drive/MyDrive/RLresearch_checkpoints"
else:
    CHECKPOINT_DIR = "./checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [2]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from envs.minatar_env import MinAtarEnv
from models.encoder import MinAtarEncoder
from agents.rainbow_agent import RainbowAgent

## Environment, encoder, and agent

The environment wrapper, encoder, and full Rainbow agent (double Q-learning, dueling architecture, categorical/C51 value distribution, prioritized replay, n-step returns, noisy-net exploration) now live in `envs/`, `models/`, and `agents/` rather than inline in this notebook -- Phase 1 (`train_spr_minatar.ipynb`) reuses the exact same modules with an SPR auxiliary head attached, and Phase 2 will reuse them again for CURL and iQRL. See those files for the implementation; this notebook is purely the Phase 0 baseline training run.

## Config, build, and train

In [6]:
GAME = "breakout"
SEED = 0

NUM_ENV_STEPS = 200_000
LEARNING_STARTS = 1_000   # env steps of random actions to warm up the replay buffer before training
LOG_EVERY = 1_000
CHECKPOINT_EVERY = 20_000

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

env = MinAtarEnv(GAME, seed=SEED)
encoder = MinAtarEncoder(in_channels=env.obs_shape[0])
agent = RainbowAgent(encoder=encoder, num_actions=env.num_actions, obs_shape=env.obs_shape, device=device)

checkpoint_path = os.path.join(CHECKPOINT_DIR, f"rainbow_{GAME}.pt")
start_step = 0
episode_returns = []
losses = []

if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    agent.load_state_dict(ckpt["agent"])
    start_step = ckpt["step"]
    episode_returns = ckpt.get("episode_returns", [])
    print(f"resumed from {checkpoint_path} at step {start_step}")
else:
    print("no checkpoint found, starting from scratch")

device: cpu


resumed from ./checkpoints\rainbow_breakout.pt at step 160000


In [7]:
ep_return = 0.0
obs = env.reset()

for step in range(start_step, NUM_ENV_STEPS):
    if step < LEARNING_STARTS:
        action = np.random.randint(env.num_actions)
    else:
        action = agent.act(obs)

    next_obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    agent.store_transition(obs, action, reward, next_obs, done)
    ep_return += reward
    obs = next_obs

    if done:
        episode_returns.append(ep_return)
        ep_return = 0.0
        obs = env.reset()

    if step >= LEARNING_STARTS:
        loss = agent.train_step()
        if loss is not None:
            losses.append(loss)

    if step % LOG_EVERY == 0 and episode_returns:
        recent = episode_returns[-20:]
        recent_loss = losses[-1] if losses else float("nan")
        print(f"step={step} episodes={len(episode_returns)} "
              f"mean_return(last20)={np.mean(recent):.2f} loss={recent_loss:.4f}")

    if step % CHECKPOINT_EVERY == 0 and step > start_step:
        torch.save(
            {"agent": agent.state_dict(), "step": step, "episode_returns": episode_returns},
            checkpoint_path,
        )
        print(f"checkpoint saved at step {step}")

torch.save(
    {"agent": agent.state_dict(), "step": NUM_ENV_STEPS, "episode_returns": episode_returns},
    checkpoint_path,
)
print("training complete, final checkpoint saved")

step=160000 episodes=3130 mean_return(last20)=6.55 loss=nan


step=161000 episodes=3144 mean_return(last20)=6.10 loss=1.4789


step=162000 episodes=3159 mean_return(last20)=5.55 loss=0.8927


step=163000 episodes=3174 mean_return(last20)=5.70 loss=1.4661


step=164000 episodes=3190 mean_return(last20)=5.40 loss=1.5151


...

step=196000 episodes=3654 mean_return(last20)=5.95 loss=1.3444


step=197000 episodes=3669 mean_return(last20)=5.55 loss=1.4267


step=198000 episodes=3684 mean_return(last20)=5.50 loss=1.3719


step=199000 episodes=3698 mean_return(last20)=6.25 loss=1.6440


training complete, final checkpoint saved


In [8]:
def moving_average(x, window=20):
    if len(x) < window:
        return x
    return np.convolve(x, np.ones(window) / window, mode="valid")

plt.figure(figsize=(8, 4))
plt.plot(moving_average(episode_returns))
plt.xlabel("episode")
plt.ylabel("return (20-episode moving average)")
plt.title(f"Rainbow DQN on MinAtar — {GAME}")
plt.tight_layout()
plt.show()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_23016\136343562.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
